# Resumable chunk qualification on a Colab T4 (slice B)

**Every cell here is a person-run step for Nate.** Nothing in this notebook signs in, stores a secret, discovers data,
or runs on its own. It is a thin driver: all training logic is the reviewed `train_sft.py` resumable mode
(slice A, #587), and every check is a reviewed script beside it.

What it qualifies: that a run cut into chunks across sessions gives the same result as one uninterrupted run, on this
runtime. It uses **invented rows only** (`qualification.py invented-rows`). Real, private or approved-TRAIN rows need
Nate's separate data-placement approval and are out of scope.

Gates that stay with Nate: the account (it must be one Nate owns and uses himself; Google's terms forbid shared
accounts), installing the lock, mounting Drive, and running any training cell.

Known limits, stated before any run:
1. **Drive locking is UNQUALIFIED.** The lease guard is proven on a local filesystem only. The probe in step 5
   measures two runtimes on one Drive folder; even a no-overlap report is evidence for review, not permission to run
   runtimes in parallel. Run one runtime at a time.
2. **fp16 on the T4, `bf16=False`.** The trainer applies the requested policy (eager attention, deterministic
   algorithms, cuDNN benchmark off, fixed cuBLAS workspace), reads it back in the training process and refuses a
   chunk on any difference. Skipped fp16 updates are counted at the optimizer step and recorded per chunk.
3. **Storage is measured, not assumed.** Each chunk needs room for three checkpoints plus a margin, sized from a
   MEASURED checkpoint: the run's own last chunk, or the one-step sizing run (step 7). With neither, the run refuses
   (NOT_MEASURED). Mount free space and the account's remote quota are recorded separately.

**After a restart or a lost runtime:** run step 1 (re-upload if `/content` was lost), step 2 if the runtime is new,
then the session setup in step 3 and step 6. Everything a later step needs (invented rows, sizing and quota records,
environment records, every committed chunk with its loss history and input trace) lives in the Drive store, not in
`/content`.

## 1. Code: upload the reviewed files (person step)
Upload these files from the reviewed commit of `docs/ai-assistant/training/` to `/content/training/` with the Files
panel: `train_sft.py`, `resumable.py`, `sft_tokenize.py`, `colab_env.py`, `qualification.py`, `drive_lock_probe.py`,
`requirements-colab-cuda.lock`. No repository credentials are used. The code hash below is bound into every chunk:
compare it with the hash in the PR before going on.

In [ ]:
%cd /content/training
import train_sft
print("code_sha256", train_sft.code_sha256())

## 2. Install the locked CUDA stack (person step, Nate's install gate)
Hash-checked and dependency-free at install time. torch's CUDA 12.6 build comes from the PyTorch index named here,
everything else from PyPI. If pip refuses a hash or a wheel, stop: the lock does not fit this runtime. Restart the
session after installing (Runtime, Restart session).

In [ ]:
# PERSON-RUN: installs packages into this Colab runtime.
!pip install --require-hashes --no-deps --index-url https://pypi.org/simple \
    --extra-index-url https://download.pytorch.org/whl/cu126 -r requirements-colab-cuda.lock

## 3. Session setup (rerun after every restart)
Mounting Drive gives this notebook's code access to the whole Drive of the signed-in account, not only the store
folder. The OAuth prompt is Nate's; no agent handles it. Each call records a NEW, uniquely named environment file, so a
restart never collides with an earlier one, and a failed command stops the cell.

In [ ]:
%cd /content/training
import json, shlex, subprocess, uuid
from pathlib import Path

def run(cmd):
    print("$", cmd)
    subprocess.run(shlex.split(cmd), check=True)

# PERSON-RUN: Google's own interactive mount prompt.
from google.colab import drive
drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/fotc-qualification")
EVIDENCE = STORE / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)
LOCK = "requirements-colab-cuda.lock"

before = set(EVIDENCE.glob("env-*.json"))
run(f"python colab_env.py --out-dir {EVIDENCE} --lock {LOCK}")
(ENV_PATH,) = set(EVIDENCE.glob("env-*.json")) - before
ENV = json.loads(ENV_PATH.read_text())
ENV_ID = ENV["environment_id"]
print(ENV_ID, ENV["gpu"], ENV["lock_check"])
assert not ENV["lock_check"]["problems"], "runtime does not match the lock"

T, K = 8, 3
DATA = EVIDENCE / "invented.jsonl"
SIZING = EVIDENCE / "sizing.json"
COMMON = f"--environment-id {ENV_ID} --environment-record {ENV_PATH} --lock {LOCK}"

def fresh_run_dir(name):
    path = Path(f"/content/run-{name}-{uuid.uuid4().hex[:8]}")
    path.mkdir()
    return path

## 4. Invented rows (stored on Drive; regenerated and checked after a loss)

In [ ]:
regen = Path(f"/content/invented-{uuid.uuid4().hex[:8]}.jsonl")
run(f"python qualification.py invented-rows --out {regen} --rows 64")
if DATA.exists():
    assert DATA.read_bytes() == regen.read_bytes(), "stored invented rows differ from the deterministic generator"
else:
    DATA.write_bytes(regen.read_bytes())
print(DATA)

## 5. Drive lock probe (optional, two runtimes at once)
Run the first cell in two runtimes at the same time, each with its own label, then the report cell in one. Any
overlap means the store is NOT QUALIFIED; no overlap is evidence for review, not permission to run in parallel.

In [ ]:
run(f"python drive_lock_probe.py run --store {STORE}/lockprobe --label runtime-a --seconds 300")

In [ ]:
run(f"python drive_lock_probe.py report --store {STORE}/lockprobe")

## 6. Remote quota (person step, every session)
Type the free space shown on the account's storage page. The mount's own free-space figure is recorded separately and
is not treated as the quota.

In [ ]:
FREE_BYTES_FROM_STORAGE_PAGE = None  # PERSON: enter the number from the provider's storage page
assert FREE_BYTES_FROM_STORAGE_PAGE, "enter the free space from the storage page"
QUOTA = EVIDENCE / f"quota-{uuid.uuid4().hex[:8]}.json"
QUOTA.write_text(json.dumps({"free_bytes": int(FREE_BYTES_FROM_STORAGE_PAGE),
                             "source": "storage page, entered by Nate"}))
print(QUOTA)

## 7. Sizing run (person step, separately approved: one training step)
Measures the real checkpoint size on this model, code, lock and runtime. It is the only run allowed without a measured
size, because it produces it. It is skipped when the store already holds a sizing record.

In [ ]:
if not SIZING.exists():
    run(f"python train_sft.py --data {DATA} --run-dir {fresh_run_dir('sizing')} --resume-store {STORE}/sizing "
        f"--run-id sizing-t4 --total-steps 1 --chunk-steps 1 --sizing-run {COMMON}")
    run(f"python qualification.py sizing-record --store {STORE}/sizing --run-id sizing-t4 --out {SIZING}")
print(json.loads(SIZING.read_text()))

## 8. Uninterrupted reference run (person step: runs training)
`T` optimizer updates in one chunk. The base model is downloaded from its pinned public revision by `train_sft.py`.

In [ ]:
run(f"python train_sft.py --data {DATA} --run-dir {fresh_run_dir('whole')} --resume-store {STORE}/whole "
    f"--run-id whole --total-steps {T} --chunk-steps {T} {COMMON} "
    f"--checkpoint-size-record {SIZING} --remote-quota-record {QUOTA}")

## 9. Chunked run, first chunk (person step: runs training)
`K` updates, then commit. Then restart the session (Runtime, Restart session) and rerun steps 3 and 6 before step 10.

In [ ]:
run(f"python train_sft.py --data {DATA} --run-dir {fresh_run_dir('chunk')} --resume-store {STORE}/chunked "
    f"--run-id chunked --total-steps {T} --chunk-steps {K} {COMMON} "
    f"--checkpoint-size-record {SIZING} --remote-quota-record {QUOTA}")

## 10. Chunked run, resume to the end (person step: runs training, in a fresh session)

In [ ]:
run(f"python train_sft.py --data {DATA} --run-dir {fresh_run_dir('chunk')} --resume-store {STORE}/chunked "
    f"--run-id chunked --total-steps {T} --chunk-steps {T - K} {COMMON} "
    f"--checkpoint-size-record {SIZING} --remote-quota-record {QUOTA}")

## 11. Compare against the gate fixed before results
Reads only the Drive store. PASS needs complete, verified evidence: both lineages verify; both runs complete at the
same bound configuration; the chunked run really resumed; equal update and skipped-update counts; the same trained-input
trace; complete loss histories 1..T within atol 1e-4 / rtol 1e-3 with identical learning rates; final adapter relative
L2 at most 1e-4. Missing evidence fails. A FAIL is investigated, never met by widening a tolerance.

In [ ]:
run(f"python qualification.py compare --whole-store {STORE}/whole --whole-run whole "
    f"--chunk-store {STORE}/chunked --chunk-run chunked")